In [0]:
# Cell 1: imports, settings and city config
import os
import time
import requests
import yaml
import xml.etree.ElementTree as ET
from datetime import date, datetime, timedelta, timezone

# Settings used by the whole notebook
CATALOG = "city_lens"
USER_AGENT = "CityLens/1.0 (portfolio project)"
PAGE_SIZE = 200          # records per SRU request (the API maximum is 1000)

# Load the list of cities from the repo's config folder.
# Notebooks in a Git folder run from their own folder, so ../ goes up to the repo root.
with open("../config/cities.yaml") as f:
    config = yaml.safe_load(f)

cities = config["cities"]

# Check: one line per city
for city in cities:
    print(city["key"], city["lat"], city["lon"], city["radius_km"])

In [0]:
# Cell 2: work out the "since" date for this run
BRONZE_TABLE = f"{CATALOG}.bronze.sru_raw"
FIRST_RUN_DAYS = 30

today = date.today()

if spark.catalog.tableExists(BRONZE_TABLE):
    # Start from the last day we already loaded (one day of overlap is fine)
    last_date = spark.sql(f"SELECT max(ingest_date) AS d FROM {BRONZE_TABLE}").first()["d"]
    since = last_date if last_date else today - timedelta(days=FIRST_RUN_DAYS)
else:
    # First run: the table doesn't exist yet
    since = today - timedelta(days=FIRST_RUN_DAYS)

since_str = since.isoformat()   # SRU wants YYYY-MM-DD
print("Table exists:", spark.catalog.tableExists(BRONZE_TABLE))
print("Fetching notices modified since", since_str)

In [0]:
# Cell 3: build the SRU query and test it
SRU_URL = "https://repository.overheid.nl/sru"

def build_query(city, since_str):
    """CQL query: official notices within radius_km of the city centre, modified since a date."""
    point = f'{city["lat"]} {city["lon"]} {city["radius_km"]:.3f}'   # lat lon radius_km
    return (
        'c.product-area=="officielepublicaties"'
        f' AND w.locatiepunt within/etrs89 "{point}"'
        f' AND dt.modified>={since_str}'
        ' sortBy dt.modified/sort.ascending'
    )

def fetch_page(query, start_record, page_size):
    """One SRU request; returns the response text (XML)."""
    params = {
        "query": query,
        "startRecord": start_record,
        "maximumRecords": page_size,
    }
    resp = requests.get(SRU_URL, params=params,
                        headers={"User-Agent": USER_AGENT}, timeout=30)
    resp.raise_for_status()
    return resp.text

# Test with the first city and a single record
city = cities[0]
query = build_query(city, since_str)
print("Query:", query)

test_xml = fetch_page(query, start_record=1, page_size=1)
root = ET.fromstring(test_xml)
total = root.find(".//{*}numberOfRecords")
print("Total matching notices:", total.text if total is not None else "not found")

In [0]:
# Cell 4: fetch all pages, save raw files, collect one row per notice
VOLUME_ROOT = f"/Volumes/{CATALOG}/bronze/raw/sru"

ingest_ts = datetime.now(timezone.utc)
rows = []

for city in cities:
    query = build_query(city, since_str)
    out_dir = f"{VOLUME_ROOT}/{city['key']}/{today.isoformat()}"
    os.makedirs(out_dir, exist_ok=True)

    start = 1
    page = 1
    total = None

    while total is None or start <= total:
        xml_text = fetch_page(query, start, PAGE_SIZE)

        # 1. Keep the raw page exactly as received
        page_file = f"{out_dir}/page_{page:04d}.xml"
        with open(page_file, "w", encoding="utf-8") as f:
            f.write(xml_text)

        # 2. Split the page into single notices
        root = ET.fromstring(xml_text)
        if total is None:
            total = int(root.find(".//{*}numberOfRecords").text)
            print(f"{city['key']}: {total} notices to fetch")

        records = root.findall(".//{*}records/{*}record")
        if not records:          # safety: stop if a page comes back empty
            break

        for rec in records:
            rows.append({
                "city_key": city["key"],
                "ingest_ts": ingest_ts,
                "ingest_date": today,
                "query": query,
                "page_file": page_file,
                "raw_xml": ET.tostring(rec, encoding="unicode"),
            })

        print(f"  page {page}: {len(records)} records (start {start})")
        start += PAGE_SIZE
        page += 1
        time.sleep(1)            # be polite to the API

print("Total rows collected:", len(rows))

In [0]:
# Cell 5: write the rows to the bronze Delta table
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DateType

schema = StructType([
    StructField("city_key",    StringType(),    False),
    StructField("ingest_ts",   TimestampType(), False),
    StructField("ingest_date", DateType(),      False),
    StructField("query",       StringType(),    True),
    StructField("page_file",   StringType(),    True),
    StructField("raw_xml",     StringType(),    False),
])

df = spark.createDataFrame(rows, schema=schema)

# Re-running on the same day replaces today's rows instead of duplicating them
if spark.catalog.tableExists(BRONZE_TABLE):
    spark.sql(f"DELETE FROM {BRONZE_TABLE} WHERE ingest_date = DATE'{today.isoformat()}'")

df.write.format("delta").mode("append").saveAsTable(BRONZE_TABLE)

print("Written to", BRONZE_TABLE)

In [0]:
# Cell 6: check
display(spark.sql(f"""
    SELECT ingest_date, city_key, count(*) AS notices
    FROM {BRONZE_TABLE}
    GROUP BY ingest_date, city_key
"""))

display(spark.table(BRONZE_TABLE).select("raw_xml").limit(2))